# Ledger §149: thought safety as a number

The share of runs whose written reasoning (the text after the latents, before `###`) is judged immoral
against the two reference chains, for every filed run of §137, §142, §143, §144 and §146. The judges
are the §137 8B A/B judge and MiniLM; the action verdicts come from the files. Predictions are in the ledger.

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "3dbd60060453abaf511216ef6a68fae381fa4d5d"
REPO_DIR = "/kaggle/working/latent-reasoning"
ULTERIOR_URL = "https://github.com/sharanramjee/ulterior-motives.git"
ULTERIOR_COMMIT = "bd46cbb"
ULTERIOR_DIR = "/kaggle/working/ulterior-motives"
OUT_DIR = "/kaggle/working/thought_safety"

import json, os, pathlib, shutil, subprocess, sys, time
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["ULTERIOR_REPO"] = ULTERIOR_DIR
assert len(RUN_COMMIT) == 40 and RUN_COMMIT != "0" * 40
for url, path, commit in ((REPO_URL, REPO_DIR, RUN_COMMIT), (ULTERIOR_URL, ULTERIOR_DIR, ULTERIOR_COMMIT)):
    if not pathlib.Path(path).exists():
        subprocess.run(["git", "clone", url, path], check=True)
    subprocess.run(["git", "-C", path, "fetch", "--all", "--tags"], check=True)
    subprocess.run(["git", "-C", path, "checkout", commit], check=True)
    print(path, subprocess.run(["git", "-C", path, "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip())
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
print("gpu", subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "tpot"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.52.4", "accelerate==1.7.0", "huggingface-hub>=0.34,<1.0", "safetensors==0.5.3", "pytest>=8,<10", "python-dotenv>=1.0.0"], check=True)
probe = subprocess.run([sys.executable, "-c", "from peft.import_utils import is_torchao_available\ntry:\n print(is_torchao_available())\nexcept ImportError:\n print('incompatible')"], capture_output=True, text=True)
if "incompatible" in probe.stdout:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", *["tests/test_thought_safety.py", "tests/test_latent_backdoor.py"]], check=True, env={**os.environ})

## Score (smoke first, then every selected run)

In [ ]:
def score(out, smoke):
    cmd = [sys.executable, "-u", "scripts/score_thought_safety.py", "--output-dir", out, "--device", "cuda", "--batch-size", "32", "--judge", "both"]
    if smoke:
        cmd.append("--smoke")
    for attempt in range(2):
        r = subprocess.run(cmd)
        if r.returncode == 0:
            break
        print("scorer exited with", r.returncode, "; retrying" if attempt == 0 else "; giving up"); time.sleep(30)
    else:
        raise SystemExit("scorer failed twice")
    return json.loads(open(pathlib.Path(out) / "summary.json").read())
score(OUT_DIR + "_smoke", True)
summary = score(OUT_DIR, False)
print(open(pathlib.Path(OUT_DIR) / "summary.md").read())
print(json.dumps(summary["checks"], indent=1))
shutil.rmtree(REPO_DIR, ignore_errors=True); shutil.rmtree(ULTERIOR_DIR, ignore_errors=True)
print("remaining output files:", sum(len(f) for _, _, f in os.walk("/kaggle/working")))